# Direction Z: Chú giải sinh học top features (CARD/ResFinder families)

**Câu hỏi:** *Model có học đúng gen kháng thật không, hay chủ yếu bám vào marker lineage/mobile?*

1. **Stability selection** trên accessory genome: chi2 top-K qua nhiều subsample → tần suất chọn (feature ổn định, không phụ thuộc 1 split).
2. **Chú giải theo catalog AMR** (họ gene CARD/ResFinder + co-resistance kim loại/biocide + mobile element) cho cả top accessory lẫn paper-ready 50.
3. Phân loại evidence: **direct AMR / indirect (co-selection, mobile) / unknown (locus tag, Roary group, số — cần sequence)**.
4. Bảng coverage + bảng feature chú giải + hình.


## 0. Imports + clone dữ liệu

In [ ]:
import re, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from IPython.display import display
from sklearn.feature_selection import chi2
import matplotlib.pyplot as plt

REPO_URL="https://github.com/347251369/Antimicrobial-resistance-prediction-in-Salmonella.git"
BASE_DIR=Path("/content/salmonella_direction_Z_annotation")
REPO_DIR=BASE_DIR/"Antimicrobial-resistance-prediction-in-Salmonella"
EXTRACT_DIR=BASE_DIR/"extracted"; OUTPUT_DIR=BASE_DIR/"outputs"
for d in [BASE_DIR,EXTRACT_DIR,OUTPUT_DIR]: d.mkdir(parents=True,exist_ok=True)
DRUGS=["AMP","AUG","AXO","CHL","FOX"]; N_SUB=25; SUBSAMPLE=0.8; TOPK=50; TOP_REPORT=40; SEED=42

In [ ]:
if not REPO_DIR.exists():
    !git clone --depth 1 "{REPO_URL}" "{REPO_DIR}"
!apt-get update -qq
!apt-get install -y unrar > /dev/null
acc_dir=EXTRACT_DIR/"accessory_gene"; acc_dir.mkdir(parents=True,exist_ok=True)
rar=REPO_DIR/"results"/"Roary"/"accessory gene existence matrix.rar"
if not any(acc_dir.glob("*")):
    !unrar x -o+ "{rar}" "{acc_dir}/" > /dev/null
!find "{acc_dir}" -maxdepth 2 -type f | head

## 1. Nạp dữ liệu

In [ ]:
def cn(df):
    o=df.copy();d=[]
    for c in list(o.columns):
        if c=="Unnamed: 0": d.append(c);continue
        if o[c].dtype=="object":
            v=pd.to_numeric(o[c],errors="coerce")
            if v.notna().mean()>0.95: o[c]=v.fillna(0)
            else: d.append(c)
    return o.drop(columns=d).fillna(0)
def pl(y):
    if isinstance(y,pd.DataFrame): y=y[y.columns[-1]]
    return pd.to_numeric(y.replace({"S":0,"I":0,"R":1,"s":0,"i":0,"r":1,"Susceptible":0,"Intermediate":0,"Resistant":1}),errors="coerce")
def find_largest(root):
    cs=[p for p in Path(root).rglob("*") if p.is_file() and p.suffix.lower() in [".csv",".tsv",".txt",".xlsx"]]
    return sorted(cs,key=lambda p:p.stat().st_size,reverse=True)[0]
ACC=cn(pd.read_csv(find_largest(acc_dir)))
def load(drug):
    dd=REPO_DIR/"data"/"csv"/drug
    X=cn(pd.read_csv(dd/"gene.csv"))
    lf=dd/f"{drug}_label.csv"
    if not lf.exists(): lf=list(dd.glob("*label*.csv"))[0]
    yf=pl(pd.read_csv(lf));m=yf.notna().values;pos=np.where(m)[0]
    return X.loc[m].reset_index(drop=True),yf.loc[m].reset_index(drop=True).astype(int),ACC.iloc[pos].reset_index(drop=True)
print("accessory:",ACC.shape)

## 2. Catalog AMR + classifier

Họ gene lấy theo CARD/ResFinder (beta-lactamase, tetracycline, sulfonamide, trimethoprim, aminoglycoside, phenicol, quinolone, macrolide, colistin, fosfomycin, efflux/regulator) + co-resistance kim loại/biocide (mer/pco/sil/ars/gol/qac) + mobile element (integrase/transposase/IS/plasmid). Tên dạng locus tag / group_ / số được đánh dấu **unknown** (cần sequence).


In [ ]:
AMR_RULES=[
 (r"^(bla|tem|shv|ctx|oxa|cmy|kpc|ndm|vim|imp|ges|per|veb|pdc|dha|acc|act|mir|mox|fox|ampc|cfx|ccr)","beta_lactam","direct","beta-lactamase"),
 (r"^tet[a-zx]?$|^tet\(|^tetr$","tetracycline","direct","tetracycline resistance"),
 (r"^(sul[123]?|folp)","sulfonamide","direct","dihydropteroate synthase / sulfonamide"),
 (r"^(dfr|dhfr)","trimethoprim","direct","dihydrofolate reductase"),
 (r"^(aac|aad|aph|ant|strab?|rmt|arma|aada|kan|neo)","aminoglycoside","direct","aminoglycoside-modifying enzyme"),
 (r"^(cat|cml|cmla?|flor?|cmr)","phenicol","direct","chloramphenicol/florfenicol resistance"),
 (r"^(qnr|oqx|qep|gyra|gyrb|parc|pare)","quinolone","direct","quinolone target/qnr/efflux"),
 (r"^(erm|mph|mef|ere)","macrolide","direct","macrolide resistance"),
 (r"^(mcr|pmra|pmrb|arna)","colistin","direct","colistin/polymyxin resistance"),
 (r"^(fosa|fosb|fosx)","fosfomycin","direct","fosfomycin resistance"),
 (r"^(acr[a-f]|tolc|mdfa|mdt[a-k]|emr[a-y]|mar[a-r]|rama?|soxs|mex[a-z]|cpxa|baer)","efflux_regulatory","direct","multidrug efflux pump / regulator"),
 (r"^(mer[a-z]?)","mercury","indirect","mercury resistance operon (co-selection)"),
 (r"^(pco[a-z]?|cop[a-z]?|cueo|cus[a-z])","copper","indirect","copper resistance (co-selection)"),
 (r"^(sil[a-z]?)","silver","indirect","silver resistance (co-selection)"),
 (r"^(ars[a-r]?)","arsenic","indirect","arsenic resistance (co-selection)"),
 (r"^(gol[a-z]?)","gold","indirect","metal efflux (co-selection)"),
 (r"^(qac|suge?|emre)","biocide","indirect","biocide/quaternary ammonium efflux (co-selection)"),
 (r"^(int[i0-9]?|integrase|tnp[a-z]?|transposase|is[0-9a-z]+|ins[a-z]|conj|tra[a-z]|rep[a-z]?|mob[a-z]?)","mobile_element","indirect","mobile genetic element / plasmid (lineage/co-transfer)"),
]
def classify(name):
    low=str(name).lower()
    if re.match(r"^[a-z0-9]+_[0-9]+$",low) and re.search(r"\d",low.split("_")[0]): return ("unknown_locus_tag","unknown","locus tag đặc trưng chủng — cần sequence")
    if re.match(r"^group_\d+$",low): return ("unknown_roary_group","unknown","Roary group không tên — cần sequence")
    if re.match(r"^\d+$",low): return ("unknown_numeric","unknown","cột số không tên — cần sequence")
    for pat,cat,ev,mech in AMR_RULES:
        if re.search(pat,low): return (cat,ev,mech)
    return ("named_non_amr","unknown","gene có tên nhưng ngoài catalog AMR")

## 3. Stability selection + chú giải

In [ ]:
rng=np.random.RandomState(SEED); rows=[]; cov=[]
for drug in DRUGS:
    Xr,y,Xa=load(drug); y=np.asarray(y); cols=np.array(Xa.columns)
    var=Xa.values.var(0); keep=np.where(var>0)[0]; Xv=Xa.values[:,keep]; colv=cols[keep]
    freq=np.zeros(len(colv)); imp=np.zeros(len(colv)); n=len(y)
    for _ in range(N_SUB):
        idx=rng.choice(n,int(SUBSAMPLE*n),replace=False); ys=y[idx]
        if len(np.unique(ys))<2: continue
        chi,_=chi2(np.clip(Xv[idx],0,None),ys); chi=np.nan_to_num(chi); top=np.argsort(chi)[::-1][:TOPK]
        freq[top]+=1; imp[top]+=chi[top]
    sel=pd.DataFrame({"drug":drug,"feature":colv,"stability":freq/N_SUB,
                      "mean_chi2":np.divide(imp,np.maximum(freq,1))})
    sel=sel[sel.stability>0].sort_values("stability",ascending=False).head(TOP_REPORT).reset_index(drop=True)
    a=sel.feature.map(classify); sel["category"]=[x[0] for x in a]; sel["evidence"]=[x[1] for x in a]; sel["mechanism"]=[x[2] for x in a]; sel["source"]="accessory_stability"
    rows.append(sel)
    cov.append({"drug":drug,"n_top":len(sel),"direct":int((sel.evidence=="direct").sum()),
        "indirect":int((sel.evidence=="indirect").sum()),"unknown":int((sel.evidence=="unknown").sum()),
        "pct_interpretable":round(100*float((sel.evidence!="unknown").mean()),1)})
    pr=pd.DataFrame({"drug":drug,"feature":[c for c in Xr.columns if c!="Unnamed: 0"]})
    pa=pr.feature.map(classify); pr["stability"]=np.nan; pr["mean_chi2"]=np.nan
    pr["category"]=[x[0] for x in pa]; pr["evidence"]=[x[1] for x in pa]; pr["mechanism"]=[x[2] for x in pa]; pr["source"]="paper_ready50"
    rows.append(pr)
ann=pd.concat(rows,ignore_index=True); ann.to_csv(OUTPUT_DIR/"Z_annotated_features.csv",index=False)
covdf=pd.DataFrame(cov); covdf.to_csv(OUTPUT_DIR/"Z_coverage_summary.csv",index=False)
display(covdf)

## 4. Feature AMR có tên (direct + indirect)

In [ ]:
print("--- Top accessory: direct AMR ---")
d=ann[(ann.source=="accessory_stability")&(ann.evidence=="direct")]
display(d[["drug","feature","stability","category","mechanism"]] if len(d) else pd.DataFrame({"msg":["none"]}))
print("--- Paper-ready 50: direct + indirect ---")
p=ann[(ann.source=="paper_ready50")&(ann.evidence.isin(["direct","indirect"]))]
display(p[["drug","feature","category","evidence","mechanism"]].drop_duplicates())

## 5. Hình: phân bố evidence

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
for ax,src,title in [(axes[0],"accessory_stability","Top-40 accessory (stability)"),(axes[1],"paper_ready50","Paper-ready 50")]:
    sub=ann[ann.source==src]
    piv=sub.groupby(["drug","evidence"]).size().unstack(fill_value=0)
    for col in ["direct","indirect","unknown"]:
        if col not in piv: piv[col]=0
    piv=piv[["direct","indirect","unknown"]]
    piv.plot(kind="bar",stacked=True,ax=ax,color=["#2a9d8f","#e9c46a","#adb5bd"])
    ax.set_title(title); ax.set_ylabel("số feature"); ax.legend(title="evidence")
plt.tight_layout(); plt.savefig(OUTPUT_DIR/"Z_fig_evidence.png",dpi=150); plt.show()

## 6. (Tùy chọn) Chạy ResFinder/RGI THẬT

Chú giải ở trên là name/catalog-based. Muốn chú giải chuẩn cần **sequence** cho representative của mỗi pangenome cluster. Nếu có FASTA (ví dụ `pan_genome_reference.fa` từ Roary, hoặc protein FASTA), bỏ comment để chạy:

```bash
# ResFinder (Center for Genomic Epidemiology)
pip install resfinder
python -m resfinder -ifa representative.fasta -o resfinder_out -acq -l 0.6 -t 0.8

# hoặc RGI (CARD)
pip install rgi
rgi main -i representative.faa -o rgi_out -t protein --clean
```

Sau đó join kết quả (cột `Resistance gene` / `Best_Hit_ARO`) vào `Z_annotated_features.csv` theo tên cluster để thay các dòng `unknown_*` bằng chú giải thật.


In [ ]:
# Placeholder: nếu có RESFINDER/RGI output, đọc và merge ở đây.
# rgi=pd.read_csv("rgi_out.txt",sep="\t"); ...  merge theo cluster id
print("Bỏ qua bước sequence-based (chưa cung cấp FASTA). Dùng chú giải catalog-based ở trên.")

## 7. Gom output

In [ ]:
import shutil
md_out=OUTPUT_DIR/"Z_SUMMARY.md"
md_out.write_text("# Direction Z — annotation coverage\n\n## Coverage (top-40 accessory stability)\n"
    +covdf.to_markdown(index=False)
    +"\n\n## Paper-ready 50: named AMR features\n"
    +ann[(ann.source=='paper_ready50')&(ann.evidence.isin(['direct','indirect']))][['drug','feature','category','evidence','mechanism']].drop_duplicates().to_markdown(index=False),
    encoding="utf-8")
shutil.make_archive(str(BASE_DIR/"direction_Z_outputs"),"zip",OUTPUT_DIR)
print("Saved:",md_out)
try:
    from google.colab import files; files.download(str(BASE_DIR/"direction_Z_outputs.zip"))
except Exception as e: print("skip:",e)

## 8. Cách viết vào khóa luận

- **Bằng chứng model học sinh học thật:** paper-ready 50 chứa gene AMR đúng cơ chế — ví dụ `floR` cho CHL (chloramphenicol/florfenicol), `sul1` cho AMP (sulfonamide), `aadA1`/`dfrA12` (integron-borne). Viết: "feature quan trọng bao gồm gene kháng đã biết, phù hợp cơ chế của thuốc".
- **Cảnh báo trung thực (điểm mạnh về học thuật):**
  - Nhiều hit là **co-selection kim loại** (`mer*`, `pco*`) và **integron/mobile** — gợi ý model một phần bám marker lineage/plasmid, không hẳn causal cho từng thuốc.
  - Top accessory features gần như **không chú giải được từ tên** (locus tag/Roary group) → cần sequence-based RGI/ResFinder. Đây là giới hạn annotation coverage, không phải lỗi model.
- **Bảng dùng:** `Z_annotated_features.csv` (drug, feature, stability, category, evidence, mechanism), `Z_coverage_summary.csv`, hình `Z_fig_evidence.png`. Cắm vào chương *Biological interpretation* và *Limitations*.
